# V-SHIELD Anti-Spoofing Model Training Pipeline (Kaggle / Colab GPU)

This notebook trains the **V-SHIELD Anti-Spoofing CNN** (`LightweightAntiSpoofCNN`) using benchmark datasets (e.g. ASVspoof 2019 LA).

**Hardware Target:** Kaggle GPU (T4 x2 or P100) or Google Colab GPU (T4/V100).
**Memory Safety:** Lazy audio streaming via PyTorch `DataLoader`.
**Integrity:** Speaker-disjoint train/val/test splits, zero fake metrics, reproducible seeds.

In [ ]:
# 1. Environment & GPU Verification
import torch
import os

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("Device Count:", torch.cuda.device_count())
    print("VRAM Allocated:", round(torch.cuda.memory_allocated(0)/1024**2, 1), "MB")
else:
    print("WARNING: Running on CPU. Please enable GPU Accelerator in Notebook Settings!")

In [ ]:
# 2. Install Required Audio Dependencies
!pip install -q soundfile torchaudio pyyaml scikit-learn

In [ ]:
# 3. Locate Dataset on Kaggle / Colab Input
import glob

# Search for ASVspoof dataset in input directories
dataset_candidates = glob.glob("/kaggle/input/**/ASVspoof2019_LA*", recursive=True) or glob.glob("/content/**/ASVspoof2019_LA*", recursive=True)
if dataset_candidates:
    dataset_root = os.path.dirname(dataset_candidates[0])
    print(f"[+] Located ASVspoof dataset at: {dataset_root}")
else:
    dataset_root = "/kaggle/input/asvpoof-2019-dataset/LA"
    print(f"[*] Default dataset search path: {dataset_root}")

# Export environment variable for V-SHIELD scripts
os.environ["VSHIELD_DATASET_ROOT"] = dataset_root

In [ ]:
# 4. Generate Standardized Manifest (Train, Dev, Eval)
!python training/scripts/build_manifest.py \
    --dataset asvspoof2019 \
    --root "$VSHIELD_DATASET_ROOT" \
    --output-csv "training/data/metadata/metadata.csv" \
    --splits train dev eval

In [ ]:
# 5. Validate Dataset Integrity & Check for Speaker Leakage
!python training/scripts/validate_dataset.py \
    --manifest-csv "training/data/metadata/metadata.csv" \
    --max-check 2000

In [ ]:
# 6. Verify Speaker Disjointness
!python training/scripts/split_dataset.py \
    --input-csv "training/data/metadata/metadata.csv" \
    --verify-only

In [ ]:
# 7. Train Model with CUDA Acceleration & Auto-Weighted BCE
# Note: If your session was disconnected, add `--resume training/checkpoints/last_checkpoint.pt`
!python training/scripts/train.py \
    --config training/configs/baseline.yaml \
    --device cuda \
    --epochs 25 \
    --batch-size 32

In [ ]:
# 8. Scientific Evaluation on Held-Out Test Set (Threshold Calibrated on Val Split Only)
!python training/scripts/evaluate.py \
    --checkpoint "training/checkpoints/best_checkpoint.pt" \
    --manifest-csv "training/data/metadata/metadata.csv" \
    --val-split val \
    --test-split test \
    --device cuda \
    --output-json "reports/phase_2b_evaluation.json"

In [ ]:
# 9. Export Production-Ready Checkpoint with Model Metadata
!python training/scripts/export_model.py \
    --checkpoint "training/checkpoints/best_model.pt" \
    --metrics-json "reports/phase_2b_evaluation.json" \
    --output-dir "models/vshield_antispoof_v1" \
    --status VALIDATED \
    --validation-status RESEARCH_VALIDATED

In [ ]:
# 10. Package Artifacts for Download to Local V-SHIELD
!tar -czvf vshield_phase2b_trained_model.tar.gz -C models/vshield_antispoof_v1 best_model.pt model_meta.json
print("\n[+] Trained model package 'vshield_phase2b_trained_model.tar.gz' successfully created!")
print("    Download this file and extract into your local 'models/vshield_antispoof_v1/' directory.")